# LeetCode 1200 — Minimum Absolute Difference bằng Python

**Link đề:** https://leetcode.com/problems/minimum-absolute-difference/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting

```python
class Solution:
    def minimumAbsDifference(self, arr: list[int]) -> list[list[int]]:
        ...
```

## Mục tiêu học
1. Hiểu vì sao **sau khi sắp xếp, cặp có hiệu nhỏ nhất luôn nằm kề nhau** — insight cốt lõi của rất nhiều bài.
2. Nắm `abs`, `sorted`, `zip(a, a[1:])`, list comprehension, `float('inf')`, list lồng nhau.
3. Đi từ brute force `O(n²)` → sort + 2 lượt → sort + 1 lượt → counting sort `O(n + R)`.
4. Trace từng bước, chứng minh, test đối chiếu với brute force.
5. Làm bài tập biến thể (tìm cặp gần nhất, Minimum Difference giữa max-min của k phần tử...).

## 1. Đề bài và ví dụ

Cho mảng số nguyên **phân biệt** `arr`. Tìm **mọi cặp** phần tử có **hiệu tuyệt đối nhỏ nhất** (so với mọi cặp trong mảng).

Trả về danh sách các cặp theo **thứ tự tăng dần** (so sánh theo cặp), mỗi cặp `[a, b]` thoả:
- `a`, `b` thuộc `arr`;
- `a < b`;
- `b - a` bằng hiệu tuyệt đối nhỏ nhất giữa hai phần tử bất kỳ của `arr`.

| `arr` | Kết quả |
|---|---|
| `[4, 2, 1, 3]` | `[[1, 2], [2, 3], [3, 4]]` (hiệu nhỏ nhất = 1) |
| `[1, 3, 6, 10, 15]` | `[[1, 3]]` (hiệu nhỏ nhất = 2) |
| `[3, 8, -10, 23, 19, -4, -14, 27]` | `[[-14, -10], [19, 23], [23, 27]]` (hiệu nhỏ nhất = 4) |

**Constraints:** `2 <= arr.length <= 10^5`; `-10^6 <= arr[i] <= 10^6`; các phần tử **khác nhau**.

## 2. Nền tảng Python cần biết

### 2.1. `abs` — giá trị tuyệt đối

`abs(x)` bỏ dấu âm. "Hiệu tuyệt đối" của `a` và `b` là `abs(a - b)` — không phụ thuộc thứ tự.

In [1]:
print(abs(-7), abs(7), abs(3 - 8), abs(8 - 3))
print("hiệu tuyệt đối giữa -14 và -10:", abs(-14 - (-10)))

7 7 5 5
hiệu tuyệt đối giữa -14 và -10: 4


### 2.2. List lồng nhau (list of lists)

Kết quả là list mà mỗi phần tử là list `[a, b]`. Python so sánh hai list **theo từng phần tử từ trái sang** (thứ tự từ điển) — đó là ý nghĩa "tăng dần theo cặp".

In [2]:
pairs = [[3, 4], [1, 2], [2, 3]]
print("pairs[0] =", pairs[0], "| pairs[0][1] =", pairs[0][1])
print("[1, 2] < [1, 3] ?", [1, 2] < [1, 3], "| [2, 0] < [1, 9] ?", [2, 0] < [1, 9])
print("sorted(pairs) =", sorted(pairs))

pairs[0] = [3, 4] | pairs[0][1] = 4
[1, 2] < [1, 3] ? True | [2, 0] < [1, 9] ? False
sorted(pairs) = [[1, 2], [2, 3], [3, 4]]


### 2.3. `sorted` và phần tử kề nhau

Sau `s = sorted(arr)`, các cặp **kề nhau** là `(s[i-1], s[i])` với `i = 1..n-1`. Có hai cách duyệt:
- dùng chỉ số: `for i in range(1, len(s))`;
- dùng `zip(s, s[1:])`: ghép `s` với chính nó dịch 1 vị trí.

In [3]:
s = sorted([4, 2, 1, 3])
print("s =", s, "| s[1:] =", s[1:])
for i in range(1, len(s)):
    print("chỉ số:", (s[i - 1], s[i]))
print("zip:", list(zip(s, s[1:])))

s = [1, 2, 3, 4] | s[1:] = [2, 3, 4]
chỉ số: (1, 2)
chỉ số: (2, 3)
chỉ số: (3, 4)
zip: [(1, 2), (2, 3), (3, 4)]


`zip(a, b)` ghép phần tử cùng vị trí thành tuple, dừng khi list ngắn hơn hết. Vì `s[1:]` ngắn hơn `s` 1 phần tử nên ta được đúng `n - 1` cặp kề nhau.

### 2.4. `float('inf')` — "vô cực" để khởi tạo giá trị nhỏ nhất

In [4]:
best = float("inf")
for d in [5, 3, 8]:
    if d < best:
        best = d
print("best =", best, "| inf > 10**18 ?", float("inf") > 10**18)

best = 3 | inf > 10**18 ? True


### 2.5. List comprehension có điều kiện

`[biểu_thức for x in iterable if điều_kiện]` — tạo list mới, chỉ giữ phần tử thoả điều kiện.

In [5]:
nums = [1, 4, 6, 7, 10]
print([x for x in nums if x % 2 == 0])
print([[a, b] for a, b in zip(nums, nums[1:]) if b - a == 1])

[4, 6, 10]
[[6, 7]]


## 3. Cách 1 — Brute force: xét mọi cặp

Ý nghĩ tự nhiên: hai vòng lặp `i < j`, tính `abs(arr[i] - arr[j])`. Lượt 1 tìm `min_diff`, lượt 2 gom cặp, cuối cùng sắp xếp kết quả.

In [6]:
def min_abs_bruteforce(arr: list[int]) -> list[list[int]]:
    n = len(arr)
    min_diff = float("inf")
    for i in range(n):
        for j in range(i + 1, n):
            min_diff = min(min_diff, abs(arr[i] - arr[j]))
    result = []
    for i in range(n):
        for j in range(i + 1, n):
            if abs(arr[i] - arr[j]) == min_diff:
                a, b = min(arr[i], arr[j]), max(arr[i], arr[j])
                result.append([a, b])
    return sorted(result)

print(min_abs_bruteforce([4, 2, 1, 3]))
print(min_abs_bruteforce([1, 3, 6, 10, 15]))
print(min_abs_bruteforce([3, 8, -10, 23, 19, -4, -14, 27]))

[[1, 2], [2, 3], [3, 4]]
[[1, 3]]
[[-14, -10], [19, 23], [23, 27]]


**Nút thắt:** số cặp `n(n-1)/2`. Với `n = 10^5` là ~5·10^9 phép tính → TLE.

In [7]:
n = 10**5
print(f"Số cặp: {n * (n - 1) // 2:,}")

Số cặp: 4,999,950,000


## 4. Insight — sau khi sắp xếp, chỉ cần xét cặp **kề nhau**

Giả sử đã sắp xếp: `s[0] < s[1] < ... < s[n-1]`. Xét cặp không kề nhau `s[i] < s[k]` với `k > i + 1`. Có một phần tử ở giữa `s[i+1]`:

```text
s[k] - s[i] = (s[k] - s[i+1]) + (s[i+1] - s[i])  >  s[i+1] - s[i]
```

(vì mảng phân biệt nên `s[k] - s[i+1] > 0`). Vậy **cặp không kề nhau luôn có hiệu lớn hơn hẳn** một cặp kề nhau nằm bên trong nó → không bao giờ là hiệu nhỏ nhất.

Kết luận: chỉ cần `n - 1` hiệu kề nhau thay vì `n(n-1)/2` cặp. Thêm nữa, duyệt từ trái sang phải trên mảng đã sắp xếp cho ra các cặp **đã đúng thứ tự tăng dần** và luôn có `a < b`.

In [8]:
s = sorted([3, 8, -10, 23, 19, -4, -14, 27])
print("sorted:", s)
diffs = [b - a for a, b in zip(s, s[1:])]
print("hiệu kề nhau:", diffs, "| min =", min(diffs))

sorted: [-14, -10, -4, 3, 8, 19, 23, 27]
hiệu kề nhau: [4, 6, 7, 5, 11, 4, 4] | min = 4


## 5. Lời giải chính — giảng từng đoạn rồi ghép

### Cách 2 — Sort + hai lượt

**Đoạn 1:** `s = sorted(arr)`.
**Đoạn 2:** `min_diff = min(b - a for a, b in zip(s, s[1:]))`.
**Đoạn 3:** gom các cặp kề nhau có `b - a == min_diff`.

In [9]:
def min_abs_two_pass(arr: list[int]) -> list[list[int]]:
    s = sorted(arr)                                         # đoạn 1
    min_diff = min(b - a for a, b in zip(s, s[1:]))         # đoạn 2
    return [[a, b] for a, b in zip(s, s[1:]) if b - a == min_diff]   # đoạn 3

for arr in [[4, 2, 1, 3], [1, 3, 6, 10, 15], [3, 8, -10, 23, 19, -4, -14, 27]]:
    print(arr, "->", min_abs_two_pass(arr))

[4, 2, 1, 3] -> [[1, 2], [2, 3], [3, 4]]
[1, 3, 6, 10, 15] -> [[1, 3]]
[3, 8, -10, 23, 19, -4, -14, 27] -> [[-14, -10], [19, 23], [23, 27]]


### Cách 3 — Sort + một lượt (reset khi gặp hiệu nhỏ hơn)

Duy trì `best` (hiệu nhỏ nhất đến giờ) và `result` (các cặp có hiệu = `best`). Với mỗi cặp kề nhau có hiệu `d`:
- `d < best` → tìm được mức mới tốt hơn: `best = d`, **xoá** `result` cũ, đặt `result = [[a, b]]`.
- `d == best` → thêm `[a, b]` vào `result`.
- `d > best` → bỏ qua.

In [10]:
def min_abs_one_pass(arr: list[int]) -> list[list[int]]:
    s = sorted(arr)
    best = float("inf")
    result = []
    for i in range(1, len(s)):
        d = s[i] - s[i - 1]
        if d < best:
            best = d
            result = [[s[i - 1], s[i]]]
        elif d == best:
            result.append([s[i - 1], s[i]])
    return result

for arr in [[4, 2, 1, 3], [1, 3, 6, 10, 15], [3, 8, -10, 23, 19, -4, -14, 27]]:
    print(arr, "->", min_abs_one_pass(arr))

[4, 2, 1, 3] -> [[1, 2], [2, 3], [3, 4]]
[1, 3, 6, 10, 15] -> [[1, 3]]
[3, 8, -10, 23, 19, -4, -14, 27] -> [[-14, -10], [19, 23], [23, 27]]


`elif` = "else if": chỉ được xét khi nhánh `if` phía trên sai.

### Cách 4 — Counting sort (tận dụng miền giá trị bị chặn)

Giá trị nằm trong `[-10^6, 10^6]` → miền `R = 2·10^6 + 1`. Đánh dấu `present[x + OFFSET] = True`, duyệt miền tăng dần để có mảng đã sắp xếp mà không cần `sorted`. Độ phức tạp `O(n + R)`.
Trong thực tế Python, `sorted` thường nhanh hơn vì vòng lặp 2 triệu bước khá chậm — đây là cách để hiểu ý tưởng, không phải để nộp. Ta viết dạng chung theo `lo..hi` thực tế của mảng để demo nhanh.

In [11]:
def min_abs_counting(arr: list[int]) -> list[list[int]]:
    lo, hi = min(arr), max(arr)
    present = [False] * (hi - lo + 1)
    for x in arr:
        present[x - lo] = True
    s = [v + lo for v in range(hi - lo + 1) if present[v]]   # mảng đã sắp xếp
    best = min(b - a for a, b in zip(s, s[1:]))
    return [[a, b] for a, b in zip(s, s[1:]) if b - a == best]

print(min_abs_counting([3, 8, -10, 23, 19, -4, -14, 27]))

[[-14, -10], [19, 23], [23, 27]]


## 6. TRACE từng bước (cách một lượt)

`arr = [3, 8, -10, 23, 19, -4, -14, 27]` → `s = [-14, -10, -4, 3, 8, 19, 23, 27]`

| i | cặp (s[i-1], s[i]) | d | best trước | Quyết định | result sau |
|:---:|---|:---:|:---:|---|---|
| 1 | (-14, -10) | 4 | inf | d < best → reset | `[[-14,-10]]` |
| 2 | (-10, -4) | 6 | 4 | d > best → bỏ | `[[-14,-10]]` |
| 3 | (-4, 3) | 7 | 4 | bỏ | không đổi |
| 4 | (3, 8) | 5 | 4 | bỏ | không đổi |
| 5 | (8, 19) | 11 | 4 | bỏ | không đổi |
| 6 | (19, 23) | 4 | 4 | d == best → thêm | `[[-14,-10],[19,23]]` |
| 7 | (23, 27) | 4 | 4 | thêm | `[[-14,-10],[19,23],[23,27]]` |

In [12]:
def trace_min_abs(arr: list[int]) -> list[list[int]]:
    s = sorted(arr)
    print("sorted:", s)
    best = float("inf")
    result = []
    for i in range(1, len(s)):
        a, b = s[i - 1], s[i]
        d = b - a
        if d < best:
            action = f"d < best({best}) -> reset"
            best = d
            result = [[a, b]]
        elif d == best:
            action = "d == best -> thêm"
            result.append([a, b])
        else:
            action = f"d > best({best}) -> bỏ qua"
        print(f"i={i} cặp=({a:3d},{b:3d}) d={d:2d} | {action:24} | result={result}")
    return result

assert trace_min_abs([3, 8, -10, 23, 19, -4, -14, 27]) == [[-14, -10], [19, 23], [23, 27]]

sorted: [-14, -10, -4, 3, 8, 19, 23, 27]
i=1 cặp=(-14,-10) d= 4 | d < best(inf) -> reset   | result=[[-14, -10]]
i=2 cặp=(-10, -4) d= 6 | d > best(4) -> bỏ qua    | result=[[-14, -10]]
i=3 cặp=( -4,  3) d= 7 | d > best(4) -> bỏ qua    | result=[[-14, -10]]
i=4 cặp=(  3,  8) d= 5 | d > best(4) -> bỏ qua    | result=[[-14, -10]]
i=5 cặp=(  8, 19) d=11 | d > best(4) -> bỏ qua    | result=[[-14, -10]]
i=6 cặp=( 19, 23) d= 4 | d == best -> thêm        | result=[[-14, -10], [19, 23]]
i=7 cặp=( 23, 27) d= 4 | d == best -> thêm        | result=[[-14, -10], [19, 23], [23, 27]]


Trace thêm một ví dụ mà **mức tốt nhất bị thay đổi nhiều lần** (cho thấy vì sao cần reset `result`):

In [13]:
assert trace_min_abs([1, 10, 15, 17, 18, 30]) == [[17, 18]]

sorted: [1, 10, 15, 17, 18, 30]
i=1 cặp=(  1, 10) d= 9 | d < best(inf) -> reset   | result=[[1, 10]]
i=2 cặp=( 10, 15) d= 5 | d < best(9) -> reset     | result=[[10, 15]]
i=3 cặp=( 15, 17) d= 2 | d < best(5) -> reset     | result=[[15, 17]]
i=4 cặp=( 17, 18) d= 1 | d < best(2) -> reset     | result=[[17, 18]]
i=5 cặp=( 18, 30) d=12 | d > best(1) -> bỏ qua    | result=[[17, 18]]


## 7. Code nộp LeetCode

### 7.1. Bản chính — sort + một lượt

In [14]:
from typing import List

class Solution:
    def minimumAbsDifference(self, arr: List[int]) -> List[List[int]]:
        arr = sorted(arr)
        best = float("inf")
        result = []
        for i in range(1, len(arr)):
            d = arr[i] - arr[i - 1]
            if d < best:
                best = d
                result = [[arr[i - 1], arr[i]]]
            elif d == best:
                result.append([arr[i - 1], arr[i]])
        return result

sol = Solution()
print(sol.minimumAbsDifference([4, 2, 1, 3]))
print(sol.minimumAbsDifference([3, 8, -10, 23, 19, -4, -14, 27]))

[[1, 2], [2, 3], [3, 4]]
[[-14, -10], [19, 23], [23, 27]]


### 7.2. Bản gọn — sort + hai lượt

In [15]:
class SolutionTwoPass:
    def minimumAbsDifference(self, arr: List[int]) -> List[List[int]]:
        arr.sort()
        best = min(b - a for a, b in zip(arr, arr[1:]))
        return [[a, b] for a, b in zip(arr, arr[1:]) if b - a == best]

assert SolutionTwoPass().minimumAbsDifference([1, 3, 6, 10, 15]) == [[1, 3]]
print("SolutionTwoPass OK")

SolutionTwoPass OK


Lưu ý: bản này dùng `arr.sort()` — **sửa trực tiếp** input. Trên LeetCode chấp nhận được và tiết kiệm bộ nhớ; ngoài đời nên cẩn thận nếu người gọi còn dùng `arr`.

### 7.3. Bản counting sort (đúng nhưng thường chậm hơn trong Python)

In [16]:
class SolutionCounting:
    def minimumAbsDifference(self, arr: List[int]) -> List[List[int]]:
        lo, hi = min(arr), max(arr)
        present = [False] * (hi - lo + 1)
        for x in arr:
            present[x - lo] = True
        result, best, prev = [], float("inf"), None
        for v in range(hi - lo + 1):
            if not present[v]:
                continue
            cur = v + lo
            if prev is not None:
                d = cur - prev
                if d < best:
                    best, result = d, [[prev, cur]]
                elif d == best:
                    result.append([prev, cur])
            prev = cur
        return result

assert SolutionCounting().minimumAbsDifference([3, 8, -10, 23, 19, -4, -14, 27]) == [[-14, -10], [19, 23], [23, 27]]
print("SolutionCounting OK")

SolutionCounting OK


`prev is not None` — so sánh với `None` dùng `is`, vì `None` là đối tượng duy nhất.

## 8. Vì sao đúng?

1. **Hiệu nhỏ nhất xuất hiện ở cặp kề nhau** (mục 4): mọi cặp không kề nhau có hiệu lớn hơn hẳn một cặp kề nằm giữa. Vậy `min` trên các cặp kề = `min` trên mọi cặp.
2. **Không bỏ sót cặp tối ưu:** một cặp đạt hiệu nhỏ nhất `m` buộc phải kề nhau (nếu không, cặp kề bên trong có hiệu < m — mâu thuẫn). Ta duyệt hết các cặp kề → gom đủ.
3. **Thứ tự đúng:** duyệt `i` tăng dần trên mảng tăng dần → `a` (phần tử đầu cặp) tăng ngặt → danh sách cặp tăng dần, và `a < b` trong mỗi cặp.

**Bất biến của cách một lượt:** sau khi xử lý xong `i`, `best` = hiệu nhỏ nhất trong các cặp kề `(s[0],s[1])..(s[i-1],s[i])`, và `result` là **đúng** các cặp kề trong đoạn đó có hiệu bằng `best`, theo thứ tự.

## 9. Độ phức tạp và so sánh

Gọi `n = len(arr)`, `R = max - min + 1` (miền giá trị, tối đa ~`2·10^6`).

| Cách | Time | Space phụ (không tính output) | Ghi chú |
|---|---|---|---|
| Brute force | `O(n²)` | `O(1)` (+ sort output) | TLE với `n = 10^5` |
| Sort + 2 lượt | `O(n log n)` | `O(n)` (sorted) | gọn, dễ đọc |
| Sort + 1 lượt | `O(n log n)` | `O(n)` | một lần duyệt sau sort |
| Counting sort | `O(n + R)` | `O(R)` | lý thuyết tuyến tính, Python thực tế chậm |

Output có thể tới `n - 1` cặp (ví dụ mảng cách đều) → bản thân output đã `O(n)`.

## 10. Bộ test

Mẫu đề, biên (`n = 2`), mảng cách đều (mọi cặp kề đều là đáp án), số âm, và đối chiếu 4 cách với brute force trên dữ liệu ngẫu nhiên.

In [17]:
impls = {
    "two_pass": min_abs_two_pass,
    "one_pass": min_abs_one_pass,
    "counting": min_abs_counting,
    "Solution": lambda a: Solution().minimumAbsDifference(list(a)),
    "TwoPass": lambda a: SolutionTwoPass().minimumAbsDifference(list(a)),
    "Counting": lambda a: SolutionCounting().minimumAbsDifference(list(a)),
}
cases = [
    ([4, 2, 1, 3], [[1, 2], [2, 3], [3, 4]]),
    ([1, 3, 6, 10, 15], [[1, 3]]),
    ([3, 8, -10, 23, 19, -4, -14, 27], [[-14, -10], [19, 23], [23, 27]]),
    ([5, -5], [[-5, 5]]),                                   # n = 2
    ([10, 0, 20, 30], [[0, 10], [10, 20], [20, 30]]),       # cách đều
    ([-1000000, 1000000], [[-1000000, 1000000]]),           # biên giá trị
    ([1, 10, 15, 17, 18, 30], [[17, 18]]),                  # best đổi nhiều lần
]
for arr, expected in cases:
    assert min_abs_bruteforce(arr) == expected
    for name, f in impls.items():
        assert f(arr) == expected, (name, arr, f(arr))
    print(f"{str(arr):42} -> {expected} PASS")

[4, 2, 1, 3]                               -> [[1, 2], [2, 3], [3, 4]] PASS
[1, 3, 6, 10, 15]                          -> [[1, 3]] PASS
[3, 8, -10, 23, 19, -4, -14, 27]           -> [[-14, -10], [19, 23], [23, 27]] PASS
[5, -5]                                    -> [[-5, 5]] PASS
[10, 0, 20, 30]                            -> [[0, 10], [10, 20], [20, 30]] PASS


[-1000000, 1000000]                        -> [[-1000000, 1000000]] PASS
[1, 10, 15, 17, 18, 30]                    -> [[17, 18]] PASS


In [18]:
import random

rng = random.Random(1200)
for _ in range(300):
    n = rng.randint(2, 40)
    arr = rng.sample(range(-60, 61), n)         # sample -> các phần tử phân biệt
    expected = min_abs_bruteforce(arr)
    for name, f in impls.items():
        assert f(arr) == expected, (name, arr)
print("300 test ngẫu nhiên: mọi cách khớp brute force")

big = rng.sample(range(-10**6, 10**6 + 1), 10**5)
r1 = Solution().minimumAbsDifference(big)
assert r1 == min_abs_two_pass(big)
print("n = 10^5: số cặp =", len(r1), "| hiệu nhỏ nhất =", r1[0][1] - r1[0][0])

300 test ngẫu nhiên: mọi cách khớp brute force


n = 10^5: số cặp = 4941 | hiệu nhỏ nhất = 1


`rng.sample(population, k)` lấy `k` phần tử **không lặp** — phù hợp với ràng buộc "phần tử phân biệt".

## 11. Lỗi người mới hay gặp

### Lỗi 1: Không reset `result` khi tìm được hiệu nhỏ hơn

In [19]:
def wrong_no_reset(arr):
    s = sorted(arr)
    best, result = float("inf"), []
    for i in range(1, len(s)):
        d = s[i] - s[i - 1]
        if d <= best:                 # SAI: gộp chung < và ==, không xoá cặp cũ
            best = d
            result.append([s[i - 1], s[i]])
    return result

print("sai :", wrong_no_reset([1, 10, 15, 17, 18, 30]))
print("đúng:", min_abs_one_pass([1, 10, 15, 17, 18, 30]))

sai : [[1, 10], [10, 15], [15, 17], [17, 18]]
đúng: [[17, 18]]


Vì sao sai: các cặp có hiệu 9, 5, 2 được thêm khi chúng **từng** là nhỏ nhất, nhưng không bị xoá khi xuất hiện mức 1. Sửa: tách `d < best` (reset) và `d == best` (thêm).

### Lỗi 2: Quên sắp xếp, chỉ xét cặp kề trong mảng gốc

In [20]:
def wrong_no_sort(arr):
    best = min(abs(b - a) for a, b in zip(arr, arr[1:]))
    return [[min(a, b), max(a, b)] for a, b in zip(arr, arr[1:]) if abs(b - a) == best]

print("sai :", wrong_no_sort([1, 5, 2]), "| đúng:", min_abs_one_pass([1, 5, 2]))

sai : [[2, 5]] | đúng: [[1, 2]]


Insight "chỉ xét cặp kề" **chỉ đúng sau khi sắp xếp**. Trong `[1, 5, 2]`, cặp `(1, 2)` không kề nhau trong mảng gốc.

### Lỗi 3: Trả tuple thay vì list, hoặc `[b, a]` ngược thứ tự

`zip` sinh tuple `(a, b)`; LeetCode thường vẫn chấp nhận nhưng để đúng kiểu `List[List[int]]` nên viết `[a, b]`. Và nhớ `a < b`.

### Lỗi 4: Khởi tạo `best = 0` thay vì vô cực

`d < 0` không bao giờ xảy ra → `result` rỗng mãi.

In [21]:
def wrong_init_zero(arr):
    s = sorted(arr)
    best, result = 0, []
    for i in range(1, len(s)):
        d = s[i] - s[i - 1]
        if d < best:
            best, result = d, [[s[i - 1], s[i]]]
        elif d == best:
            result.append([s[i - 1], s[i]])
    return result

print("best=0 ->", wrong_init_zero([4, 2, 1, 3]), "(rỗng, sai)")

best=0 -> [] (rỗng, sai)


## 12. Mẹo nhận dạng pattern

- Đề hỏi **"hiệu nhỏ nhất / cặp gần nhau nhất"** trên **một mảng số** (1 chiều) → **sort rồi xét phần tử kề nhau**.
- "Liệt kê mọi cặp đạt giá trị tối ưu" → kỹ thuật **một lượt với reset** (`<` thì reset, `==` thì thêm).
- Miền giá trị nhỏ → có thể thay sort bằng counting sort / bucket.

Bài cùng họ: 530/783 Minimum Distance Between BST Nodes (in-order = mảng đã sort), 1984 Minimum Difference Between Highest and Lowest of K Scores (sort + cửa sổ), 539 Minimum Time Difference (sort + nhớ cặp vòng quanh), 2148/164 Maximum Gap.

## 13. Biến thể

### 13.1. Chỉ cần giá trị hiệu nhỏ nhất

In [22]:
def min_gap(arr):
    s = sorted(arr)
    return min(b - a for a, b in zip(s, s[1:]))

print(min_gap([3, 8, -10, 23, 19, -4, -14, 27]))

4


### 13.2. Minimum Difference Between Highest and Lowest of K Scores (LeetCode 1984)

Chọn `k` phần tử sao cho `max - min` nhỏ nhất. Sau khi sort, `k` phần tử tốt nhất luôn là một **đoạn liên tiếp** — tổng quát hoá của "cặp kề nhau" (k = 2).

In [23]:
def minimum_difference_k(nums: list[int], k: int) -> int:
    s = sorted(nums)
    return min(s[i + k - 1] - s[i] for i in range(len(s) - k + 1))

print(minimum_difference_k([90], 1), minimum_difference_k([9, 4, 1, 7], 2))   # 0 2

0 2


### 13.3. Minimum Time Difference (LeetCode 539) — có "vòng"

Thời gian `"HH:MM"` trên đồng hồ 24h: ngoài cặp kề nhau còn phải xét cặp **cuối → đầu** vòng qua nửa đêm: `first + 1440 - last`.

In [24]:
def find_min_difference(times: list[str]) -> int:
    mins = sorted(int(t[:2]) * 60 + int(t[3:]) for t in times)
    best = min(b - a for a, b in zip(mins, mins[1:])) if len(mins) > 1 else 1440
    return min(best, mins[0] + 1440 - mins[-1])

print(find_min_difference(["23:59", "00:00"]), find_min_difference(["00:00", "23:59", "00:00"]))   # 1 0

1 0


## 14. Bài tập (TODO — không gọi khi chạy)

**Bài 1 (tính tay).** `arr = [40, 11, 26, 27, -20]`. Sắp xếp, ghi các hiệu kề nhau, đáp án.

**Bài 2 (dự đoán output).** `list(zip([1, 2, 3], [1, 2, 3][1:]))`, `[1, 9] < [2, 0]`, `abs(-3 - 4)`.

**Bài 3 (điền TODO).** Hoàn thiện cách hai lượt.

In [25]:
def exercise_min_abs(arr: list[int]) -> list[list[int]]:
    s = sorted(arr)
    # TODO 1: tính best = hiệu nhỏ nhất giữa các phần tử kề nhau của s
    # TODO 2: trả list các [a, b] kề nhau có b - a == best
    pass

# assert exercise_min_abs([4, 2, 1, 3]) == [[1, 2], [2, 3], [3, 4]]

**Bài 4 (viết lại).** Viết cách **một lượt** bằng `zip` (không dùng chỉ số).

In [26]:
def exercise_min_abs_one_pass(arr: list[int]) -> list[list[int]]:
    # TODO
    pass

# assert exercise_min_abs_one_pass([3, 8, -10, 23, 19, -4, -14, 27]) == [[-14, -10], [19, 23], [23, 27]]

**Bài 5 (biến thể).** Cho **hai** mảng `a`, `b`. Tìm `min |a[i] - b[j]|`. Gợi ý: sort cả hai, hai con trỏ — luôn tiến con trỏ đang trỏ giá trị **nhỏ hơn**.

In [27]:
def exercise_min_diff_two_arrays(a: list[int], b: list[int]) -> int:
    # TODO
    pass

# assert exercise_min_diff_two_arrays([1, 3, 15, 11, 2], [23, 127, 235, 19, 8]) == 3

**Bài 6 (mini challenge).** Mảng **có thể có phần tử trùng** (khác đề gốc). Trả các cặp `[a, b]` **khác nhau** (theo giá trị) có hiệu nhỏ nhất. Ví dụ `[1, 1, 2]` → hiệu nhỏ nhất 0 → `[[1, 1]]`.

In [28]:
def exercise_min_abs_with_dups(arr: list[int]) -> list[list[int]]:
    # TODO
    pass

# assert exercise_min_abs_with_dups([1, 1, 2]) == [[1, 1]]
# assert exercise_min_abs_with_dups([3, 1, 3, 1]) == [[1, 1], [3, 3]]
# assert exercise_min_abs_with_dups([5, 1, 3]) == [[1, 3], [3, 5]]

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Sort trước | Viết dãy hiệu kề | hiệu nhỏ nhất xuất hiện mấy lần? |
| 2 | zip dừng ở list ngắn | so sánh list theo phần tử đầu | `abs` bỏ dấu |
| 3 | Hiệu kề nhau | `zip(s, s[1:])` | `min(b - a for a, b in ...)` |
| 4 | Reset khi `<`, thêm khi `==` | `for a, b in zip(s, s[1:])` | `best, res = d, [[a, b]]` |
| 5 | Giống merge hai mảng đã sort | `i, j = 0, 0` | `if a[i] < b[j]: i += 1 else: j += 1` |
| 6 | Có trùng → hiệu 0 có thể xảy ra | cặp kề vẫn đúng sau sort | khử trùng bằng `set` của tuple rồi sort, hoặc kiểm tra cặp trước |

## 16. Đáp án tham khảo

**Bài 1:** sort → `[-20, 11, 26, 27, 40]`, hiệu `31, 15, 1, 13` → `[[26, 27]]`.

**Bài 2:** `[(1, 2), (2, 3)]`, `True`, `7`.

In [29]:
def answer_min_abs(arr):
    s = sorted(arr)
    best = min(b - a for a, b in zip(s, s[1:]))
    return [[a, b] for a, b in zip(s, s[1:]) if b - a == best]

def answer_min_abs_one_pass(arr):
    s = sorted(arr)
    best, res = float("inf"), []
    for a, b in zip(s, s[1:]):
        d = b - a
        if d < best:
            best, res = d, [[a, b]]
        elif d == best:
            res.append([a, b])
    return res

def answer_min_diff_two_arrays(a, b):
    a, b = sorted(a), sorted(b)
    i = j = 0
    best = float("inf")
    while i < len(a) and j < len(b):
        best = min(best, abs(a[i] - b[j]))
        if a[i] < b[j]:
            i += 1
        else:
            j += 1
    return best

def answer_min_abs_with_dups(arr):
    s = sorted(arr)
    best = min(b - a for a, b in zip(s, s[1:]))
    seen = set()
    res = []
    for a, b in zip(s, s[1:]):
        if b - a == best and (a, b) not in seen:
            seen.add((a, b))
            res.append([a, b])
    return res

assert answer_min_abs([40, 11, 26, 27, -20]) == [[26, 27]]
assert (list(zip([1, 2, 3], [1, 2, 3][1:])), [1, 9] < [2, 0], abs(-3 - 4)) == ([(1, 2), (2, 3)], True, 7)
assert answer_min_abs([4, 2, 1, 3]) == [[1, 2], [2, 3], [3, 4]]
assert answer_min_abs_one_pass([3, 8, -10, 23, 19, -4, -14, 27]) == [[-14, -10], [19, 23], [23, 27]]
assert answer_min_diff_two_arrays([1, 3, 15, 11, 2], [23, 127, 235, 19, 8]) == 3
assert answer_min_abs_with_dups([1, 1, 2]) == [[1, 1]]
assert answer_min_abs_with_dups([3, 1, 3, 1]) == [[1, 1], [3, 3]]
assert answer_min_abs_with_dups([5, 1, 3]) == [[1, 3], [3, 5]]
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 17. Template nhớ nhanh và checklist submit

```python
s = sorted(arr)
best, res = float("inf"), []
for a, b in zip(s, s[1:]):
    d = b - a
    if d < best:   best, res = d, [[a, b]]   # mức mới -> reset
    elif d == best: res.append([a, b])       # cùng mức -> thêm
return res
```

**Câu thần chú:** *"Gần nhau nhất → sort → chỉ nhìn hàng xóm."*

### Checklist trước khi submit
- [ ] Đã **sắp xếp** trước khi xét cặp kề?
- [ ] `best` khởi tạo là vô cực (hoặc tính bằng `min` riêng)?
- [ ] Tách `<` (reset) và `==` (thêm)?
- [ ] Mỗi cặp `[a, b]` có `a < b` và danh sách tăng dần?
- [ ] Không dùng hai vòng lặp lồng nhau (`n` tới `10^5`)?